# Segmentation

**AI2 · Computer Vision** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

We have previously learned about Object Detection, which allows us to locate objects in the image by predicting their *bounding boxes*. However, for some tasks we do not only need bounding boxes, but also more precise object localization. This task is called  **segmentation**.


Segmentation can be viewed as **pixel classification**, whereas for **each** pixel of image we must predict its class (*background* being one of the classes). There are two main segmentation algorithms:

* **Semantic segmentation** only tells the pixel class, and does not make a distinction between different objects of the same class
* **Instance segmentation** divides classes into different instances.

For instance segmentation, these sheep are different objects, but for semantic segmentation all sheep are represented by one class.

![Illustration](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/12-Segmentation/images/instance_vs_semantic.jpeg)

> Image from [this blog post](https://nirmalamurali.medium.com/image-classification-vs-semantic-segmentation-vs-instance-segmentation-625c33a08d50)

There are different neural architectures for segmentation, but they all have the same structure. In a way, it is similar to the autoencoder you learned about previously, but instead of deconstructing the original image, our goal is to deconstruct a **mask**. Thus, a segmentation network has the following parts:

* **Encoder** extracts features from input image
* **Decoder** transforms those features into the **mask image**, with the same size and number of channels corresponding to the number of classes.

![Illustration](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/12-Segmentation/images/segm.png)

> Image from [this publication](https://arxiv.org/pdf/2001.05566.pdf)

We should especially mention the loss function that is used for segmentation. When using classical autoencoders, we need to measure the similarity between two images, and we can use mean square error (MSE) to do that. In segmentation, each pixel in the target mask image represents the class number (one-hot-encoded along the third dimension), so we need to use loss functions specific for classification - cross-entropy loss, averaged over all pixels. If the mask is binary - **binary cross-entropy loss** (BCE) is used.

> ✅ One-hot encoding is a way to encode a class label into a vector of length equal to the number of classes. Take a look at [this article](https://datagy.io/sklearn-one-hot-encode/) on this technique.



## Segmentation for Medical Imaging

In this lesson, we will see the segmentation in action by training the network to recognize human nevi (also known as moles) on medical images. We will be using [PH<sup>2</sup> Database](https://www.fc.up.pt/addi/ph2%20database.html) of dermoscopy images as the image source. This dataset contains 200 images of three classes: typical nevus, atypical nevus, and melanoma. All images also contain a corresponding **mask** that outlines the nevus.

> ✅ This technique is particularly appropriate for this type of medical imaging, but what other real-world applications could you envision?

![navi](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/12-Segmentation/images/navi.png)

> Image from the PH<sup>2</sup> Database

We will train a model to segment any nevus from its background.



## Exercises: Semantic Segmentation

Open the notebooks below to learn more about different semantic segmentation architectures, practice working with them, and see them in action.

* [Semantic Segmentation Pytorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/12-Segmentation/SemanticSegmentationPytorch.ipynb)
* [Semantic Segmentation TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/12-Segmentation/SemanticSegmentationTF.ipynb)



## Conclusion

Segmentation is a very powerful technique for image classification, moving beyond bounding boxes to pixel-level classification. It is a technique used in medical imaging, among other applications.



## Challenge

Body segmentation is just one of the common tasks that we can do with images of people. Another important tasks include **skeleton detection** and **pose detection**. Try out [OpenPose](https://github.com/CMU-Perceptual-Computing-Lab/openpose) library to see how pose detection can be used.



## Review & Self Study

This [wikipedia article](https://wikipedia.org/wiki/Image_segmentation) offers a good overview of the various applications of this technique. Learn more on your own about the subdomains of Instance segmentation and Panoptic segmentation in this field of inquiry.



## Assignment

In this lab, try **human body segmentation** using [Segmentation Full Body MADS Dataset](https://www.kaggle.com/datasets/tapakah68/segmentation-full-body-mads-dataset) from Kaggle.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Mask to bounding box

●○○ · ~8 min

> A segmentation mask implies a detection box: the tight box around the foreground pixels is [min col, min row, max col + 1, max row + 1] in xyxy convention — the '+1' makes x2, y2 exclusive so the box actually covers the extreme pixel. np.nonzero hands you the coordinates directly. Decide the empty-mask convention up front; here it is None.

Implement `mask_to_bbox(mask)` for a binary (H, W) mask of 0/1 → `[x1, y1, x2, y2]` (list of ints) tightly enclosing all 1-pixels: x1 = leftmost column, y1 = topmost row, x2 = one past the rightmost column, y2 = one past the bottom row. If the mask has no foreground pixels, return `None`.

<details><summary>Hint</summary>

ys, xs = np.nonzero(mask); if xs.size == 0 → None; else [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1] as ints.

</details>


In [ ]:
import numpy as np

def mask_to_bbox(mask):
    m = np.asarray(mask)
    # your code
    return [0, 0, 0, 0]

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    h, w = int(rng.integers(3, 13)), int(rng.integers(3, 13))\n    p = float(rng.choice([0.0, 0.2, 0.4, 0.7]))\n    return {\"mask\": (rng.random((h, w)) < p).astype(int)}\n\ndef mask_to_bbox(mask):\n    m = np.asarray(mask)\n    ys, xs = np.nonzero(m)\n    if xs.size == 0:\n        return None\n    return [int(xs.min()), int(ys.min()),\n            int(xs.max()) + 1, int(ys.max()) + 1]", "mask_to_bbox", 40)

### Task 2 — Pixel accuracy + Dice

●●○ · ~10 min

> Two metrics dominate binary segmentation: pixel accuracy — the fraction of correctly classified pixels, which is misleading under class imbalance — and Dice = 2·|P∩G| / (|P| + |G|), effectively the F1 score of masks. The edge case everyone forgets: two empty masks are a perfect match, so Dice = 1.

Implement `seg_metrics(pred, gt)` for binary (H, W) masks of 0/1 → `{'pixel_acc': float, 'dice': float}`. Dice = 2·|P∩G| / (|P| + |G|); if both masks are empty, dice = 1.0.

<details><summary>Hint</summary>

np.logical_and(pred, gt).sum() gives the intersection; compute the denominator first and branch on it being 0.

</details>


In [ ]:
import numpy as np

def seg_metrics(pred, gt):
    p = np.asarray(pred)
    g = np.asarray(gt)
    # your code
    return {"pixel_acc": 0.0, "dice": 0.0}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    h, w = int(rng.integers(3, 11)), int(rng.integers(3, 11))\n    p = float(rng.uniform(0.15, 0.6))\n    pred = (rng.random((h, w)) < p).astype(int)\n    gt = (rng.random((h, w)) < p).astype(int)\n    # force empty-mask edge cases ~30% of the time — the\n    # both-empty → dice=1 rule must actually be exercised\n    u = rng.random()\n    if u < 0.10:\n        pred[:] = 0; gt[:] = 0\n    elif u < 0.20:\n        gt[:] = 0\n    elif u < 0.30:\n        pred[:] = 0\n    return {\"pred\": pred, \"gt\": gt}\n\ndef seg_metrics(pred, gt):\n    p = np.asarray(pred)\n    g = np.asarray(gt)\n    acc = float((p == g).mean())\n    inter = float(np.logical_and(p, g).sum())\n    s = float(p.sum() + g.sum())\n    dice = 1.0 if s == 0.0 else 2.0 * inter / s\n    return {\"pixel_acc\": acc, \"dice\": dice}", "seg_metrics", 40)

### Task 3 — Count blobs (connected components)

●●● · ~45 min

> Counting objects in a binary mask is the connected-components problem: every unvisited foreground cell starts a new blob, then a flood fill marks everything connected to it. Flood fill: push the cell onto a stack; while the stack is non-empty pop a cell, and for each of its 4 neighbors that is in-bounds, a `1`, and unvisited — mark it visited and push it. With 4-connectivity only up/down/left/right neighbors join a blob — diagonal contact does not count. Plain Python loops are plenty for tiny grids; don't reach for scipy.

Implement `count_blobs(grid)`: number of connected components of 1-cells in a binary (H, W) grid under 4-connectivity (diagonals do NOT connect). Return a plain int.

<details><summary>Hint</summary>

Keep a `seen` mask; scan cells, and when you hit an unvisited 1, increment the counter and BFS/DFS from it, marking visited neighbors.

</details>


In [ ]:
import numpy as np

def count_blobs(grid):
    g = np.asarray(grid)
    # your code — flood-fill / union-find connected components
    return int(g.sum())  # wrong — counts cells, not blobs

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    h, w = int(rng.integers(3, 11)), int(rng.integers(3, 11))\n    p = float(rng.choice([0.2, 0.35, 0.5, 0.65]))\n    return {\"grid\": (rng.random((h, w)) < p).astype(int)}\n\ndef count_blobs(grid):\n    g = np.asarray(grid)\n    h, w = g.shape\n    seen = np.zeros((h, w), dtype=bool)\n    cnt = 0\n    for i in range(h):\n        for j in range(w):\n            if g[i, j] and not seen[i, j]:\n                cnt += 1\n                seen[i, j] = True\n                stack = [(i, j)]\n                while stack:\n                    x, y = stack.pop()\n                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                        nx, ny = x + dx, y + dy\n                        if (0 <= nx < h and 0 <= ny < w\n                                and g[nx, ny] and not seen[nx, ny]):\n                            seen[nx, ny] = True\n                            stack.append((nx, ny))\n    return cnt", "count_blobs", 40)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/12-Segmentation)
- [Yandex ML handbook (RU): convolutional networks](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/convolutional-neural-networks)